# Part 5 · 01 Raw cache

The only notebook in part 5 that talks to EDGAR. It downloads the chosen document of every event once more and keeps its **raw bytes**, undecoded, in `cache/raw/raw_YYYY.zip`, with one line per fetch in `cache/raw_manifest.parquet`. `fetch_releases.py` kept only its cleaned text, so the HTML has to be fetched again — but only this once: cleaning then runs on the cache and never returns to EDGAR.

**Which document:** the one `fetch_releases.py` chose. With `extraction_report_full.csv` the script goes straight to it; without it, it reads each index page again and applies `fetch_releases.py`'s own rules (`parse_index`, `to_text`, `looks_like_prose`).

**Rerun-safe:** archives are committed every 500 events; after the kernel stops, rerun the setup cell and the same pull cell, and only the missing events are fetched. The SEC allows 10 requests a second with a descriptive User-Agent; the script uses 9.

**Before running:** run `00_setup_check` once.

In [ ]:
%pip install -q beautifulsoup4 lxml nltk pyarrow datasketch tiktoken transformers


In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

# Notebook paths.
CODE = STEP5_CODE
STEP5_DIR.mkdir(parents=True, exist_ok=True)

for sub in ["cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
sys.path.insert(0, str(CODE))

missing = [f for f in ["settings.py", "cache_raw.py"] if not (CODE / f).exists()]
assert not missing, f"Missing code in {CODE}: {missing}"
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

## Test on 200 events

One to two minutes, with a progress line every 10 events (events, documents found, failures, speed, time left). Then check that the sources look like `fetch_releases.py`'s (mostly EX-99.1) before the full pass.

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'cache_raw.py'), '--limit', '200'], check=True)

In [ ]:
import pandas as pd
import settings as S

man = pd.read_parquet(S.RAW_MANIFEST)
docs = man[man.kind == "doc"]
old = pd.read_csv(S.EXTRACTION_REPORT).source.fillna("")
print(f"events fetched: {len(docs):,} | documents stored: {(docs.sha256 != '').sum():,}")
print(docs.source.value_counts().to_string())
for label, new, was in [("EX-99.1", docs.source.str.startswith("EX-99.1"), old.str.startswith("EX-99.1")),
                        ("other EX-99", docs.source.str.startswith("EX-99") & ~docs.source.str.startswith("EX-99.1"),
                         old.str.startswith("EX-99") & ~old.str.startswith("EX-99.1")),
                        ("8-K body", docs.source.eq("8-K body"), old.eq("8-K body")),
                        ("no prose document", docs.source.eq("none"), old.eq("none"))]:
    print(f"  {label:<18} now {100 * new.mean():5.1f}%   fetch_releases {100 * was.mean():5.1f}%")

## Full pass

About 116,000 events: 3 to 4 hours with the full report, 7 to 8 hours without. **Agree with the team before running it**, so the pass is made once. Set `RUN_FULL = True`; if the kernel stops, rerun the setup cell and this one.

A download that fails for a passing reason (no answer, SEC refusal, server error) is marked `fetch failed`, never `none`, and is tried again whenever this cell is rerun. At the end, rerun the cell until the `failed` count is 0 or stops falling; the check cell above shows what is left.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    __import__("subprocess").run([sys.executable, str(CODE / "cache_raw.py")], check=True)
else:
    print("full pass not run: set RUN_FULL = True")